# Render the Kannada audio on a Colab GPU

Building a whole voice is a couple of hours on a laptop. On a Colab GPU it is a lot less,
and it is not your machine doing it.

**Before you start:** pick a **GPU** runtime. Runtime -> Change runtime type -> T4 GPU, or L4/A100
if you have them.

Not a TPU. PyTorch only reaches a TPU through `torch_xla`, which this code does not use, so a TPU
runtime quietly falls back to that machine's CPU and runs slower than the GPU options.

**Two secrets**, set with the key icon in the left sidebar, then toggle *Notebook access* on:

| Name | What | Where to get it |
|---|---|---|
| `HF_TOKEN` | reads the gated voice model | huggingface.co/settings/tokens (read scope) |
| `GH_TOKEN` | pushes the clips back, optional | github.com/settings/tokens (repo scope) |

Accept the model terms once at <https://huggingface.co/ai4bharat/indic-parler-tts>, or the
download will 401.

Without `GH_TOKEN` the notebook still works; it just hands you a zip at the end instead of pushing.


## 1. Check you actually got a GPU


In [ ]:
import torch

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU. '
                     'A TPU runtime will not work: this code uses CUDA, not torch_xla, '
                     'so it would silently run on the CPU.')


## 2. Install

Takes a few minutes. The git-lfs line is needed because one dependency is cloned from GitHub.


In [ ]:
!apt-get -qq install -y ffmpeg git-lfs > /dev/null
!pip -q install numpy 'git+https://github.com/huggingface/parler-tts.git' 2>&1 | tail -2
print('installed')


## 3. Sign in to Hugging Face


In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get('HF_TOKEN'))
print('signed in as', whoami()['name'])


## 4. Get the repo

Point `REPO` at your fork. `BRANCH` is the branch the clips should land on.


In [ ]:
REPO   = 'karthik4222/this-blr-namma-bengaluru'
BRANCH = 'kannada-two-voices'

import os
from google.colab import userdata

gh = None
try:
    gh = userdata.get('GH_TOKEN')
except Exception:
    pass

url = f'https://{gh}@github.com/{REPO}.git' if gh else f'https://github.com/{REPO}.git'
!rm -rf work
!git clone -q --branch {BRANCH} {url} work
assert os.path.isdir('work'), f'could not clone {BRANCH} from {REPO} - check the branch name and GH_TOKEN'
os.chdir('/content/work')
!git log --oneline -1


## 5. Render

Only clips that are missing get made, so re-running is cheap. Use `--voice Anu` to do one
voice, or leave it off for every voice the script is configured with. Add `--force` to redo
everything from scratch.


In [ ]:
# sanity check: are we in the right tree?
!ls docs/audio/kn | head
!python scripts/build-kannada-audio.py --check || true

!python scripts/build-kannada-audio.py --voice Anu


## 6. Check nothing is out of order

Fails if a phrase is missing a clip in any voice, has two clips, or points at a file that is not there.


In [ ]:
!python scripts/build-kannada-audio.py --check


## 7. Take the clips

Run **7a** to push straight back to the branch, or **7b** to download a zip and commit it yourself.


### 7a. Push


In [ ]:
!git config user.name  'colab'
!git config user.email 'colab@users.noreply.github.com'
!git add docs/audio/kn
!git commit -q -m 'Render Kannada audio on a Colab GPU' || echo 'nothing to commit'
!git push origin HEAD:{BRANCH}


### 7b. Or download a zip


In [ ]:
from google.colab import files
!cd /content/work && zip -qr /content/kannada-audio.zip docs/audio/kn
files.download('/content/kannada-audio.zip')


---

### A thing worth knowing

A faster GPU gets you the same audio sooner, not better audio. The model samples with a fixed
seed, so the same phrase gives byte-identical output on any hardware. If a clip comes out wrong
on your laptop it will come out equally wrong here; that is a job for the retry logic in the
build script, not for more compute.


---

## Optional: try different ways of asking, when a voice adds words

Anu was dropped because she inserted words the text did not contain, even on clips that
ran a perfectly normal length. Sampled decoding is the usual cause: the model picks
randomly at each step and can wander off the text. Greedy decoding removes that, at the
cost of a flatter delivery.

This renders the same words four ways so you can listen and judge. It changes nothing in
the repo.


In [ ]:
import numpy as np, torch, IPython.display as ipd
from parler_tts import ParlerTTSConfig, ParlerTTSForConditionalGeneration
from transformers import AutoConfig, AutoTokenizer, set_seed

MODEL_ID = 'ai4bharat/indic-parler-tts'
SPEAKER  = 'Anu'
WORDS    = ['ಎಷ್ಟು?', 'ಗೊತ್ತಿಲ್ಲ', 'ಇಲ್ಲ', 'ಪೆಕೋಸ್']

try: AutoConfig.register('parler_tts', ParlerTTSConfig)
except ValueError: pass

model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID).to('cuda').eval()
ptok  = AutoTokenizer.from_pretrained(MODEL_ID, config=model.config)
stok  = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
SR    = model.config.sampling_rate

ELABORATE = (f'{SPEAKER} speaks in a casual, conversational tone, as if talking to a friend '
             'in everyday speech rather than reading aloud. The delivery is natural and '
             'expressive at a moderate pace. Very high quality recording, no background noise.')
PLAIN = f'{SPEAKER} speaks clearly at a moderate pace. Very high quality recording, no background noise.'

CONFIGS = [
    ('A  sampled + rich prompt  (what shipped)', ELABORATE, {}),
    ('B  greedy  + rich prompt',                 ELABORATE, {'do_sample': False}),
    ('C  sampled + plain prompt',                PLAIN,     {}),
    ('D  greedy  + plain prompt',                PLAIN,     {'do_sample': False}),
]

for text in WORDS:
    print('=' * 60); print(text)
    for label, desc, kw in CONFIGS:
        set_seed(0)
        d = stok(desc, return_tensors='pt').to('cuda')
        p = ptok(text, return_tensors='pt').to('cuda')
        with torch.no_grad():
            g = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                               prompt_input_ids=p.input_ids,
                               prompt_attention_mask=p.attention_mask, **kw)
        a = g.cpu().numpy().squeeze().astype(np.float32)
        print(f'  {label}   {a.size/SR:.2f}s')
        ipd.display(ipd.Audio(a, rate=SR))


If **B** or **D** says only the word and nothing more, Anu is usable: set `do_sample=False`
in `load_voice` in the build script, re-render her, and the switcher can ship.

If all four still add words, it is the speaker rather than the settings, and Suresh alone
is the right answer.
